In [1]:
!pip install scikit-posthocs -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.0/47.0 kB 1.0 MB/s eta 0:00:00


In [2]:
import pandas as pd, numpy as np, itertools
from scipy.stats import kruskal
import scikit_posthocs as sp

CSV = "/kaggle/input/datasets/siam2461/3d-result-fixed/detections_3d_fixed.csv"
OUT = "/kaggle/working"

df = pd.read_csv(CSV)
print(f"detections: {len(df)}   images: {df['image_name'].nunique()}   classes: {df['class_name'].nunique()}")

desc = (df.groupby("class_name")["Z_relative"]
          .agg(n="size", mean="mean", Q1=lambda s: s.quantile(.25),
               median="median", Q3=lambda s: s.quantile(.75), std="std")
          .sort_values("median", ascending=False).round(3))
desc["IQR"] = (desc["Q3"] - desc["Q1"]).round(3)
print("\n--- class-wise relative depth ---")
print(desc.to_string())

groups = [g["Z_relative"].values for _, g in df.groupby("class_name")]
H, p = kruskal(*groups)
N, k = len(df), df["class_name"].nunique()
eps2 = (H - k + 1) / (N - k)
size = "negligible" if eps2 < .01 else "small" if eps2 < .08 else "moderate" if eps2 < .26 else "large"
print("\n--- Kruskal-Wallis ---")
print(f"H = {H:.3f}   df = {k-1}   p = {p:.3e}")
print(f"epsilon-squared = {eps2:.4f}  ({size} effect)")

dunn = sp.posthoc_dunn(df, val_col="Z_relative", group_col="class_name", p_adjust="holm")
print("\n--- Dunn post-hoc (Holm-adjusted p) ---")
print(dunn.round(4).to_string())

sig = [(a, b, dunn.loc[a, b]) for a, b in itertools.combinations(dunn.columns, 2)
       if dunn.loc[a, b] < .05]
sig.sort(key=lambda x: x[2])
print(f"\nsignificant pairs: {len(sig)} / {k*(k-1)//2}")
for a, b, pv in sig:
    print(f"  {a:22} vs {b:22}  p = {pv:.2e}")

desc.to_csv(f"{OUT}/depth_descriptive_by_class.csv")
dunn.round(6).to_csv(f"{OUT}/dunn_posthoc_holm.csv")
with open(f"{OUT}/kruskal_wallis_result.txt", "w") as f:
    f.write(f"Kruskal-Wallis H = {H:.3f}, df = {k-1}, p = {p:.3e}\n"
            f"epsilon-squared = {eps2:.4f} ({size} effect)\n"
            f"N = {N} detections, k = {k} classes\n"
            f"significant pairs after Holm correction: {len(sig)} of {k*(k-1)//2}\n")
print("\nsaved: depth_descriptive_by_class.csv, dunn_posthoc_holm.csv, kruskal_wallis_result.txt")

detections: 2903   images: 1493   classes: 8

--- class-wise relative depth ---
                        n   mean     Q1  median     Q3    std    IQR
class_name                                                          
Blight                434  0.687  0.587   0.743  0.857  0.219  0.270
brown_spot            376  0.638  0.453   0.674  0.862  0.238  0.409
corn_rust             345  0.623  0.490   0.631  0.783  0.195  0.293
downy_mildew          342  0.596  0.487   0.612  0.720  0.185  0.233
grey_leaf_spot        457  0.616  0.471   0.586  0.805  0.206  0.334
corn_smut             429  0.572  0.464   0.580  0.677  0.162  0.213
maize-streak-disease   77  0.562  0.482   0.531  0.662  0.136  0.180
healthy               443  0.481  0.413   0.489  0.569  0.148  0.156

--- Kruskal-Wallis ---
H = 310.009   df = 7   p = 4.403e-63
epsilon-squared = 0.1047  (moderate effect)

--- Dunn post-hoc (Holm-adjusted p) ---
                      Blight  brown_spot  corn_rust  corn_smut  downy_mildew  grey_l